# Introduction

This notebook is to show how to run a nextflow job that reads AOU genomic data. A similar job can be seen in dsub or cromwell notebooks.

**Note**
The nextflow UI will not work for AOU users, due to the internet connection is blocked.

The default nextflow version is 25.10, as of 07/07/2026.

Nextflow 26.04 introduces a strict syntax parser, explicit workflow contracts, and static typing with native records to catch coding bugs before execution. However, because this workbench currently defaults to Nextflow 25.10, your legacy .nf pipeline files will continue to work normally without requiring any immediate modifications.

If you prefer the latest 26.04 release, you will need to rewrite your legacy pipeline code to accommodate the new explicit workflow contracts and strict type constraints.

More info can be found from the official nextflow webpaes:

https://nf-co.re/blog/2026/parameter-types

https://docs.seqera.io/nextflow/migrations/26-04

https://docs.seqera.io/nextflow/strict-syntax


In [ ]:
# check the default version
!nextflow -v

In [ ]:
# then we can skip this line
# can't install to the current auto-mounted paths
# Download and install to /home/jupyter/
# !curl -s -L https://github.com/nextflow-io/nextflow/releases/download/v25.10.4/nextflow -o /home/jupyter/nextflow

In [ ]:
# !ls /home/jupyter/nextflow

In [ ]:
# !chmod +x /home/jupyter/nextflow

In [ ]:
# check version
# !/home/jupyter/nextflow -v

In [ ]:
!echo ${PET_SA_EMAIL}

**Run the setup notebooks to get some env variables**

In [ ]:
!ls /home/jupyter/workspace/aou-tutorial-notebooks/Setting_Env_Variables.*

In [ ]:
# or use relative path
!ls ../../Setting_Env_Variables.*

In [ ]:
%run ../../Setting_Env_Variables_p2.py

In [ ]:
import os
my_bucket=os.environ.get('WORKSPACE_BUCKET')
my_bucket

**Write a config file for v25.10**

In [ ]:
%%bash
cat > /home/jupyter/nextflow.conf <<EOF
profiles {
  gcb {
      process.executor = "google-batch"
      process.container = "gcr.io/google-containers/ubuntu-slim:0.14"
      workDir = "$WORKSPACE_BUCKET/workflows/nextflow-scratch"

      google.location = "us-central1"
      google.zone = "us-central1-a"
      google.project = "$GOOGLE_CLOUD_PROJECT"
      google.enableRequesterPaysBuckets = true
      google.batch.debug = true
      google.batch.serviceAccountEmail = "${PET_SA_EMAIL}"
      google.batch.network = "global/networks/network"
      google.batch.subnetwork = "regions/us-central1/subnetworks/subnetwork"
      google.batch.usePrivateAddress = true
      google.batch.copyImage = "gcr.io/google.com/cloudsdktool/cloud-sdk:alpine"
      google.batch.bootDiskSize = "20.GB"
  }
}
EOF

In [ ]:
!cat /home/jupyter/nextflow.conf

**Write a nf file for v25.10**

In [ ]:
%%writefile test_mount.nf

params.plinkDir = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed"

chromosomes = Channel.of(21)

input_bed = chromosomes.map { "${params.plinkDir}/clinvar.chr${it}.bed"}
input_bim = chromosomes.map { "${params.plinkDir}/clinvar.chr${it}.bim"}
input_fam = chromosomes.map { "${params.plinkDir}/clinvar.chr${it}.fam"}

process test_mount {
    container 'biocontainer/plink2:alpha2.3_jan2020'
          
    input:
    val chr
    path input_bed
    path input_bim
    path input_fam
            
    output:
    path "test11.txt"
    path "help_plink.txt"
    path "vm_cpu.txt"
    path "vm_ram.txt"
    path "vm_disk.txt"
    path "path_clinvar.txt"
    path "path2_clinvar.txt"    
   path "path_clinvar_chr2_fam.txt"
    path "path2_clinvar_chr1_fam.txt"


    // New outputs for input paths
    path "local_bed_path.txt"
    path "local_bim_path.txt"
    path "local_fam_path.txt"
    
    script:
    """
    echo hello world > test11.txt
    nproc > vm_cpu.txt
    cat /proc/meminfo | grep MemTotal > vm_ram.txt
    df -h > vm_disk.txt
    plink2 --help > help_plink.txt
    
    # Capture the absolute local paths for the inputs
    readlink -f ${input_bed} > local_bed_path.txt
    readlink -f ${input_bim} > local_bim_path.txt
    readlink -f ${input_fam} > local_fam_path.txt

    # Existing mount checks
    ln -s /mnt/disks/vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed clinvar
    ls clinvar > path_clinvar.txt
    cat clinvar/chr2.fam |head > path_clinvar_chr2_fam.txt
    
    ls /mnt/disks/vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed > path2_clinvar.txt
    cat /mnt/disks/vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed/clinvar.chr1.fam |head > path2_clinvar_chr1_fam.txt
    

    """
}

workflow {    
    test_mount(chromosomes, input_bed, input_bim, input_fam)
}

workflow.onComplete {
    println ( workflow.success ? """
        Pipeline execution summary
        ---------------------------
        Completed at: ${workflow.complete}
        Duration    : ${workflow.duration}
        Success     : ${workflow.success}
        workDir     : ${workflow.workDir}
        exit status : ${workflow.exitStatus}
        """ : """
        Failed: ${workflow.errorReport}
        exit status : ${workflow.exitStatus}
        """
    )
}

**Run the nf file**

In [ ]:
!rm test_mount.log

In [ ]:
!nextflow -log test_mount.log run test_mount.nf -c /home/jupyter/nextflow.conf -profile gcb

**write a script to extract info from the log file**

In [ ]:
import re
import subprocess

def get_workdirs_from_log(log_file, run_gcloud=False):
    """
    Parse Nextflow log and extract unique workDir GCS paths.
    Optionally list contents using gcloud storage.

    Args:
        log_file (str): path to log file
        run_gcloud (bool): whether to run `gcloud storage ls`

    Returns:
        List[str]
    """
    workdirs = []

    with open(log_file, "r") as f:
        for line in f:
            match = re.search(r'workDir:\s+(gs://[^\s\]]+)', line)
            if match:
                workdirs.append(match.group(1))

    # deduplicate
    workdirs = list(set(workdirs))

    if not workdirs:
        print("No workDir paths found.")
        return []

    print(f"Found {len(workdirs)} workDir(s):")

    for path in workdirs:
        print(path)

        if run_gcloud:
            print(f"  -> gcloud storage ls {path}")
            result = subprocess.run(
                ["gcloud", "storage", "ls", path],
                capture_output=True,
                text=True
            )

            if result.returncode == 0:
                print(result.stdout)
            else:
                print("ERROR:", result.stderr)

    return workdirs

In [ ]:
log_file="test_mount.log"

In [ ]:
workdirs=get_workdirs_from_log(log_file, run_gcloud=True)

In [ ]:
output_path = workdirs[0]
output_path

In [ ]:
!gcloud storage cat {output_path}/vm_cpu.txt

In [ ]:
!gcloud storage cat {output_path}/vm_ram.txt

In [ ]:
!gcloud storage cat {output_path}/vm_disk.txt

In [ ]:
!gcloud storage cat {output_path}/local_bed_path.txt

**You can see the mounted bucket path as well**

In [ ]:
!gcloud storage cat {output_path}/path_clinvar.txt |head

In [ ]:
# !gcloud storage cat {output_path}/path_clinvar_chr2_fam.txt 

In [ ]:
#!gcloud storage cat {output_path}/path2_clinvar_chr1_fam.txt 

**Notes**

As of 07/06/2026, While the proprietary Fusion v2 file system is not yet fully supported or enabled within the restricted All of Us (AoU) Workbench environment, Nextflow achieves a functionally similar result through its interaction with the platform's native infrastructure.

When a Nextflow process defines a gs:// path as an input, it acts as a system-level trigger that prompts the underlying Terra/GCP environment to mount the corresponding Google Cloud Storage bucket via gcsfuse.

This 'Just-In-Time' mounting eliminates the costly download/upload bottleneck typical of other workflow engines like Cromwell or dsub. By creating this virtualized link to the /mnt/disks/ directory, Nextflow allows genomic tools (such as samtools or plink2) to stream data on-demand. This dramatically speeds up I/O-heavy workflows—such as CRAM slicing or VCF filtering—and prevents 'Disk Full' errors by reducing the need for local data staging on the VM.

Examples are shown below. Users are free to test.

Tip: To successfully 'trigger' the mount, ensure your Nextflow process includes at least one small file from the target bucket (like a txt file) in the input: block. This signals the environment to open the connection to that bucket.

**Warning**

While using mounted path is convenient, misuse could lead to extremely slow jobs and unexpectedly high costs.

You may consider avoiding the mounted path when running tools that perform heavy or random file access directly on mounted paths, including but not limited to:

❌ PLINK / PLINK 2.0 (--bfile, --pfile, --score)

❌ Any GWAS / PRS workflows reading .bed, .bgen, .pgen

❌ Parallel jobs reading the same files from the mount

❌ Repeatedly opening the same large files across many tasks

Please read these links for more details:

https://docs.cloud.google.com/storage/docs/cloud-storage-fuse/overview?authuser=1

https://oneuptime.com/blog/post/2026-02-17-how-to-mount-cloud-storage-buckets-as-file-systems-in-google-cloud-batch-jobs/view

Quote: "Mounting Cloud Storage buckets in Google Cloud Batch jobs with GCS FUSE lets you use file-based processing code without rewriting it for object storage APIs. The key is understanding the performance characteristics: GCS FUSE works well for sequential reads of large files and streaming writes, but poorly for random access patterns with many small files. For the best of both worlds, use GCS FUSE to stage data to and from Cloud Storage, and local SSD for the actual processing. Always use task-specific output paths to avoid write conflicts, and tune the mount options for your specific access pattern."